In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent
sys.path.append(str(PROJECT_ROOT))

In [5]:
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings

In [3]:
from pathlib import Path
from utility.embeddings import (
    ingest_files_from_blob_urls_create_embeddings,
)

# ============================================================
# Project Configuration
# ============================================================

project_id = "G105000008"
first_name = "Yogesh"

# Source blob URLs
blob_urls = [
    "https://stintertekesusdev.blob.core.windows.net/stintertekesusdev-blob/Documents/G105000008/source_documents/CFE-4LB011-E%20(1)%20(1).pdf",
    "https://stintertekesusdev.blob.core.windows.net/stintertekesusdev-blob/Documents/G105000008/source_documents/PastedGraphic-1.png",
]

# ============================================================
# Vector Store Container Names
# ============================================================

# Text embeddings container
text_container = f"vectorstorecontainer_new_itk_text_{first_name}_{project_id}"

# Image embeddings container
image_container = f"vectorstorecontainer_new_itk_image_{first_name}_{project_id}"

# ============================================================
# Base Directories
# ============================================================

BASE_DIR = Path.cwd()

# Main data directory
DATA_DIR = BASE_DIR / "data"

# Project-specific directory
project_data_dir = DATA_DIR / project_id

# Directory where downloaded source files will be stored
DOWNLOAD_DIR = DATA_DIR / "src_files"

# ============================================================
# Input Files
# ============================================================

# Input DOCX template
INPUT_DOCX_PATH = BASE_DIR / "data" / "input_files" / "input.docx"

# Base PTA JSON configuration
BASE_PTA_JSON_PATH = BASE_DIR / "data" / "input_files" / "pta_final_6_3_1.json"

# ============================================================
# Output Files
# ============================================================

# Final generated DOCX output
OUTPUT_DOCX = project_data_dir / "final_output.docx"

# Final generated JSON output
OUTPUT_JSON = project_data_dir / "final_output.json"

# ============================================================
# Embedding Ingestion
# ============================================================

# Download files from Azure Blob Storage and create embeddings
ingest_files_from_blob_urls_create_embeddings(
    DOWNLOAD_DIR,
    blob_urls,
    project_id,
    text_container,
    image_container,
)

Running in LOCAL mode – using .env
AOAI_ENDPOINT https://oai-intertek-esus2-dev.openai.azure.com/
######### download_dir ######## d:\code\InterTek-AI-Repo\Backend\experiments\data\src_files
######## blob_urls ######### ['https://stintertekesusdev.blob.core.windows.net/stintertekesusdev-blob/Documents/G105000008/source_documents/CFE-4LB011-E%20(1)%20(1).pdf', 'https://stintertekesusdev.blob.core.windows.net/stintertekesusdev-blob/Documents/G105000008/source_documents/PastedGraphic-1.png']
######## project_id ######### G105000008
######## textDB_container_name ######### vectorstorecontainer_new_itk_text_Yogesh_G105000008
######## imageDB_container_name ######### vectorstorecontainer_new_itk_image_Yogesh_G105000008
→ Using existing database: ragdatabase_new_itk
✔ Container exists: vectorstorecontainer_new_itk_text_Yogesh_G105000008
All documents deleted successfully!
[INFO] Processing URL #0: https://stintertekesusdev.blob.core.windows.net/stintertekesusdev-blob/Documents/G105000008/sourc

{'project_id': 'G105000008',
 'image_urls': ['https://stintertekesusdev.blob.core.windows.net/stintertekesusdev-blob/Documents/G105000008/source_documents/PastedGraphic-1.png',
  'https://stintertekesusdev.blob.core.windows.net/stintertekesusdev-blob/Documents/G105000008/pdf_images/CFE-4LB011-E__1___1_.pdf/page_1.png'],
 'pdf_paths': ['D:\\code\\InterTek-AI-Repo\\Backend\\data\\G105000008\\src_files\\CFE-4LB011-E (1) (1).pdf'],
 'downloaded_pdfs': ['D:\\code\\InterTek-AI-Repo\\Backend\\data\\G105000008\\src_files\\CFE-4LB011-E (1) (1).pdf'],
 'converted_pdfs': [],
 'image_page_metadata': [{'pdf_file': 'CFE-4LB011-E (1) (1).pdf',
   'page': 1,
   'local_image_path': 'D:\\code\\InterTek-AI-Repo\\Backend\\data\\G105000008\\src_files\\CFE-4LB011-E (1) (1).pdf_page_1.png',
   'text_length': 1909,
   'raster_area': 193892,
   'vector_ops': 43,
   'blocks': 43}],
 'chunks_count': 3}

In [6]:
import os
from dotenv import load_dotenv

load_dotenv(verbose=True, override=True)

# Load environment variables
AOAI_ENDPOINT = os.getenv("aoai-endpoint")
AOAI_KEY = os.getenv("aoai-key")
API_VERSION = os.getenv("api-version")
EMBED_DEPLOY = os.getenv("embed-deploy")
COSMOS_DB_TEXT = os.getenv("cosmos-db-text")
COSMOS_CONT_TEXT = os.getenv("cosmos-cont-text")
AZURE_CONN_STRING = os.getenv("azure-conn-string")
BLOB_CONTAINER = os.getenv("blob-container")
COSMOS_URL = os.getenv("cosmos-url")
COSMOS_KEY = os.getenv("cosmos-key")
CHAT_DEPLOY = os.getenv("chat-deploy")
BLOB_CONT_NAME = os.getenv("BLOB_CONT_NAME")
COSMOS_DB_IMAGE = os.getenv("cosmos-db-image")
COSMOS_CONT_IMAGE = os.getenv("cosmos-cont-image")
ENABLE_CAD_SCHEMATICS = os.getenv("enable-cad-schematics")


In [7]:
# GPT 4.1
llm = AzureChatOpenAI(
    azure_endpoint=AOAI_ENDPOINT,
    api_key=AOAI_KEY,
    openai_api_version=API_VERSION,
    azure_deployment=CHAT_DEPLOY,
    temperature=0.1,
).with_config({"response_format": "json_object"})

In [ ]:
from utility.trf_report.trf_generation import (
    build_vectorstore_image,
    build_vectorstore_text,
    build_rag_image_pipeline_grey,
    build_vision_message_grey,
    attach_supporting_refs_grey,
    generator_evaluator_agent,
)

vs = build_vectorstore_text(text_container)
vs2 = build_vectorstore_image(image_container)

retriever = vs.as_retriever(search_kwargs={"k": 5})
image_retriever_agent = vs2.as_retriever(search_kwargs={"k": 5})

rag_image = build_rag_image_pipeline_grey(
    retriever, llm, build_vision_message_grey, attach_supporting_refs_grey, vs
)

run_single_task_tool = generator_evaluator_agent(rag_image)

print("\n===============================================")
print("       STEP 1-6 - SINGLE JSON GENERATION")
print("===============================================")

base_name = INPUT_DOCX_PATH.stem
output_json_path = project_data_dir / f"{base_name}_output.json"
excel_output_path = project_data_dir / f"{base_name}.xlsx"

print(f"[INFO] Processing single JSON: {INPUT_DOCX_PATH}")
print("[PROOF] There is no loop over multiple JSON files.")



       STEP 1-6 - SINGLE JSON GENERATION
[INFO] Processing single JSON: d:\code\InterTek-AI-Repo\Backend\experiments\data\input_files\input.docx
[PROOF] There is no loop over multiple JSON files.


In [ ]:
import json

with open(
    r"D:\code\InterTek-AI-Repo\Backend\data\input_files\pta_final_6_3_1.json",
    "r",
    encoding="utf-8",
) as f:
    data = json.load(f)

In [18]:
from utility.trf_report.trf_generation import (
    build_tasks_with_custom_prompt_grey,
    update_tasks_with_top5_images,
)

tasks, item_refs = build_tasks_with_custom_prompt_grey(data, blob_urls)

image_retriever_agent = vs2.as_retriever(search_kwargs={"k": 5})
tasks = update_tasks_with_top5_images(tasks, image_retriever_agent)


[START] Processing 794 tasks with 10 parallel threads.

[INFO] Processing task 1/794 → Mention the name of the client.
[INFO] Processing task 2/794 → Mention the address of the client.
[INFO] Processing task 3/794 → Mention the name of the product.
[INFO] Processing task 4/794 → Mention the name of the client.
[INFO] Processing task 5/794 → Mention the name of the Manufacturer.
[INFO] Processing task 6/794 → Mention the model variants of the equipment.
[INFO] Processing task 7/794 → Provide the ratings of the equipment.
[INFO] Processing task 8/794 → List the countries names addressed in summary of compliance with National Differences without '[]'. Provide me only the the country names in ISO 3166 codes as output.
[INFO] Processing task 9/794 → Classify the equipment to one of the following categories : Measurement / Control / Laboratory  by analyzing the provided product manual and product requirement document in the TRF Text and input images.
[INFO] Processing task 10/794 → Provide t

In [17]:
# GPT 4.1
llm = AzureChatOpenAI(
    azure_endpoint=AOAI_ENDPOINT,
    api_key=AOAI_KEY,
    openai_api_version="2025-01-01-preview",
    azure_deployment="gpt-5.4",
    temperature=0.1,
).with_config({"response_format": "json_object"})